### Imports

In [198]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [199]:
import pandas as pd 
import os 
import yaml
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from typing import Any, Mapping


In [200]:
from qbt.strategies.strategy_registry import create_strategy, available_strategies 
from qbt.data.dataloader import DataAdapter, DefaultDataAdapter

from qbt.core.types import ModelInputs, RunSpec

In [201]:
style_path = os.path.join(os.getcwd(), 'styler.mplstyle')
plt.style.use(style_path)

In [202]:
def load_data():
    path = os.path.join(os.getcwd(), '..', 'data', 'gold', 'freq=1D', 'tag=experiment', 'table.parquet')
    df = pd.read_parquet(path)
    df.set_index('session_date', inplace=True)
    return df

In [203]:
df = load_data()

In [204]:
df

,XLE_asof_utc,XLE_close,XLE_high,XLE_index,XLE_low,XLE_open,XLE_ret_cc,XLE_ret_co,XLE_ret_oc,XLE_ret_oo,XLE_rv,XLE_rvol,XLE_volume,DCOILWTICO,DHHNGSP,GASREGCOVW,OVXCLS
session_date,,,,,,,,,,,,,,,,,
2021-02-22 00:00:00+00:00,2021-02-22 21:00:00+00:00,19.79,20.080,0,19.240,19.260,NaN,NaN,0.027146,NaN,0.000266,0.016316,1902838,61.67,3.16,2.549,38.57
2021-02-23 00:00:00+00:00,2021-02-23 21:00:00+00:00,20.12,20.180,1,19.280,20.000,0.016538,0.010556,0.005982,0.037702,0.000837,0.028929,1305486,61.66,2.94,2.549,39.75
2021-02-24 00:00:00+00:00,2021-02-24 21:00:00+00:00,20.85,20.960,2,20.100,20.250,0.035640,0.006440,0.029199,0.012423,0.000370,0.019238,1215368,63.21,2.80,2.549,39.58
2021-02-25 00:00:00+00:00,2021-02-25 21:00:00+00:00,20.43,21.010,3,20.300,21.010,-0.020350,0.007645,-0.027994,0.036844,0.000388,0.019696,1481856,63.43,2.72,2.549,40.58
2021-02-26 00:00:00+00:00,2021-02-26 21:00:00+00:00,19.95,20.210,4,19.370,20.130,-0.023775,-0.014793,-0.008982,-0.042787,0.000939,0.030650,2106704,61.55,2.66,2.549,42.49
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-02-18 00:00:00+00:00,2026-02-18 21:00:00+00:00,54.78,54.855,1252,54.230,54.535,0.018888,0.014406,0.004482,-0.000550,0.000094,0.009672,2742576,65.33,2.98,2.790,51.49
2026-02-19 00:00:00+00:00,2026-02-19 21:00:00+00:00,55.17,55.870,1253,54.925,55.360,0.007094,0.010532,-0.003438,0.015015,0.000125,0.011162,2112428,66.66,3.09,2.790,56.73
2026-02-20 00:00:00+00:00,2026-02-20 21:00:00+00:00,54.89,55.280,1254,54.515,55.020,-0.005088,-0.002723,-0.002366,-0.006161,0.000120,0.010958,2075772,66.69,3.15,2.790,56.12


In [ ]:
df.index.min()

Timestamp('1970-01-01 00:00:00+0000', tz='UTC')

In [ ]:
df.index.max()

Timestamp('2026-02-24 00:00:00+0000', tz='UTC')

### XLE Companies

In [ ]:

path = os.path.join(os.getcwd(), '..', 'data', 'external', 'holdings-xle-2-26-26.csv')
xle = pd.read_csv(path)


In [ ]:
xle

,Name,Ticker,Identifier,SEDOL,Weight,Sector,Shares Held,Local Currency
0,EXXON MOBIL CORP,XOM,30231G102,2326618,23.811213,-,59780973.00,USD
1,CHEVRON CORP,CVX,166764100,2838555,17.308917,-,35050923.00,USD
2,CONOCOPHILLIPS,COP,20825C104,2685717,6.792925,-,22884112.00,USD
3,WILLIAMS COS INC,WMB,969457100,2967181,4.633501,-,23110361.00,USD
4,SLB LTD,SLB,806857108,2779201,4.462036,-,32317295.00,USD
5,EOG RESOURCES INC,EOG,26875P101,2318024,3.812521,-,11737764.00,USD
6,KINDER MORGAN INC,KMI,49456B101,B3NQ4P8,3.754470,-,42351682.00,USD
7,BAKER HUGHES CO,BKR,05722G100,BDHLTQ5,3.714883,-,21346431.00,USD
8,VALERO ENERGY CORP,VLO,91913Y100,2041364,3.605583,-,6598078.00,USD
9,PHILLIPS 66,PSX,718546104,B78C4Y8,3.563322,-,8716136.00,USD


### Candidate State Variables 

In [ ]:
macro_cols   = ['OVXCLS', 'DCOILWTICO', 'DHHNGSP', 'GASREGCOVW']

In [ ]:
def load_tickers(symbols):

    base_path = os.path.join(os.getcwd(), '..', 'data', 'silver', 'freq=1d')

    chunks = []
    for sym in symbols:
        path = os.path.join(base_path, f"symbol={sym}")
        chunks.append(pd.read_parquet(path))
        
    df = pd.concat(chunks)           
    df = (
        df.pivot(index="datetime", columns="ticker", values="close")
        .reset_index()
        )

    return df.rename(columns={'datetime': 'session_date'}).set_index('session_date')
    

In [ ]:
macro = load_tickers(macro_cols)

In [ ]:
print(macro)

ticker                     DCOILWTICO  DHHNGSP  GASREGCOVW  OVXCLS
session_date                                                      
2021-02-22 00:00:00+00:00       61.67     3.16       2.549   38.57
2021-02-23 00:00:00+00:00       61.66     2.94       2.549   39.75
2021-02-24 00:00:00+00:00       63.21     2.80       2.549   39.58
2021-02-25 00:00:00+00:00       63.43     2.72       2.549   40.58
2021-02-26 00:00:00+00:00       61.55     2.66       2.549   42.49
...                               ...      ...         ...     ...
2026-02-20 00:00:00+00:00       66.69     3.15       2.790   56.12
2026-02-21 00:00:00+00:00       66.69     3.15       2.790   56.12
2026-02-22 00:00:00+00:00       66.69     3.15       2.790   56.12
2026-02-23 00:00:00+00:00       66.36     3.13       2.796   59.05
2026-02-24 00:00:00+00:00         NaN      NaN         NaN   58.82

[1829 rows x 4 columns]


In [ ]:
macro.index

DatetimeIndex(['2021-02-22 00:00:00+00:00', '2021-02-23 00:00:00+00:00',
               '2021-02-24 00:00:00+00:00', '2021-02-25 00:00:00+00:00',
               '2021-02-26 00:00:00+00:00', '2021-02-27 00:00:00+00:00',
               '2021-02-28 00:00:00+00:00', '2021-03-01 00:00:00+00:00',
               '2021-03-02 00:00:00+00:00', '2021-03-03 00:00:00+00:00',
               ...
               '2026-02-15 00:00:00+00:00', '2026-02-16 00:00:00+00:00',
               '2026-02-17 00:00:00+00:00', '2026-02-18 00:00:00+00:00',
               '2026-02-19 00:00:00+00:00', '2026-02-20 00:00:00+00:00',
               '2026-02-21 00:00:00+00:00', '2026-02-22 00:00:00+00:00',
               '2026-02-23 00:00:00+00:00', '2026-02-24 00:00:00+00:00'],
              dtype='datetime64[ns, UTC]', name='session_date', length=1829, freq=None)

In [ ]:
dfr = pd.merge(df, macro, left_index=True, right_index=True)

In [ ]:
dfr

,ticker,close,open,high,low,volume,rv,rvol,asof_utc,ret_cc,ret_oo,ret_co,ret_oc,DCOILWTICO,DHHNGSP,GASREGCOVW,OVXCLS
session_date,,,,,,,,,,,,,,,,,
2021-02-22 00:00:00+00:00,XLE,19.79,19.260,20.080,19.240,1902838.0,0.000266,0.016316,2021-02-22 21:00:00+00:00,NaN,NaN,NaN,0.027146,61.67,3.16,2.549,38.57
2021-02-23 00:00:00+00:00,XLE,20.12,20.000,20.180,19.280,1305486.0,0.000837,0.028929,2021-02-23 21:00:00+00:00,0.016538,0.037702,0.010556,0.005982,61.66,2.94,2.549,39.75
2021-02-24 00:00:00+00:00,XLE,20.85,20.250,20.960,20.100,1215368.0,0.000370,0.019238,2021-02-24 21:00:00+00:00,0.035640,0.012423,0.006440,0.029199,63.21,2.80,2.549,39.58
2021-02-25 00:00:00+00:00,XLE,20.43,21.010,21.010,20.300,1481856.0,0.000388,0.019696,2021-02-25 21:00:00+00:00,-0.020350,0.036844,0.007645,-0.027994,63.43,2.72,2.549,40.58
2021-02-26 00:00:00+00:00,XLE,19.95,20.130,20.210,19.370,2106704.0,0.000939,0.030650,2021-02-26 21:00:00+00:00,-0.023775,-0.042787,-0.014793,-0.008982,61.55,2.66,2.549,42.49
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-02-18 00:00:00+00:00,XLE,54.78,54.535,54.855,54.230,2742576.0,0.000094,0.009672,2026-02-18 21:00:00+00:00,0.018888,-0.000550,0.014406,0.004482,65.33,2.98,2.790,51.49
2026-02-19 00:00:00+00:00,XLE,55.17,55.360,55.870,54.925,2112428.0,0.000125,0.011162,2026-02-19 21:00:00+00:00,0.007094,0.015015,0.010532,-0.003438,66.66,3.09,2.790,56.73
2026-02-20 00:00:00+00:00,XLE,54.89,55.020,55.280,54.515,2075772.0,0.000120,0.010958,2026-02-20 21:00:00+00:00,-0.005088,-0.006161,-0.002723,-0.002366,66.69,3.15,2.790,56.12


### Back Testing

In [ ]:
def rolling_windows(df, train_years=2, test_years=2, step_days=1):
    end_date = df.index.max()
    start_date = df.index.min()

    splits = []

    current_end = end_date

    while True:
        test_start = current_end - pd.DateOffset(years=test_years)
        train_start = test_start - pd.DateOffset(years=train_years)

        if train_start < start_date:
            break

        Train = df.loc[train_start:test_start]
        Test  = df.loc[test_start:current_end]

        splits.append((Train, Test))

        current_end -= pd.DateOffset(days=step_days)

    return splits

In [ ]:
def train(df, model):
    splits = rolling_windows(df)
    for train, test in splits:
        model.fit(train)
        model.predict(test)


In [ ]:
def build_statesignal(*, state_var: str, params: Mapping[str, Any]) -> dict[str, Any]:
    spec = {
        "state_var": str(state_var),
        "min_frac": float(params.get("min_frac", 0.10)),
        "ann_factor": int(params.get("ann_factor", 252)),
        "gamma": float(params.get("gamma", 5.0)),
        # canonical name: weight_type
        "weight_type": str(params.get("weight_type", params.get("weight_allocation", "binary"))),
        "w_min": float(params.get("w_min", 0.0)),
        # canonical name: w_max
        "w_max": float(params.get("w_max", params.get("w_high", 3.0))),
        "eps": float(params.get("eps", 1e-12)),
    }


    # minimal validation (catch silent config issues early)
    if spec["w_min"] > spec["w_max"]:
        raise ValueError(f"w_min ({spec['w_min']}) cannot exceed w_max ({spec['w_max']})")
    if spec["ann_factor"] <= 0:
        raise ValueError("ann_factor must be positive")
    if spec["min_frac"] <= 0 or spec["min_frac"] > 1:
        raise ValueError("min_frac must be in (0, 1]")

    spec = RunSpec(strategy_name='experiment', universe='xle', params=params)
    model = create_strategy('StateSignal')
    model.parse_params(spec)
    return model, spec

In [ ]:
params = {'weight_type': 'binary'}

In [ ]:
model = build_statesignal(state_var='rvol', params=params)

ValueError: StateSignalModel requires params['state_var'].

In [ ]:
ModelInputs(dfr[['ret_cc']], dfr[['rvol'] + macro_cols])

ModelInputs(ret=                             ret_cc
session_date                       
2021-02-22 00:00:00+00:00       NaN
2021-02-23 00:00:00+00:00  0.016538
2021-02-24 00:00:00+00:00  0.035640
2021-02-25 00:00:00+00:00 -0.020350
2021-02-26 00:00:00+00:00 -0.023775
...                             ...
2026-02-18 00:00:00+00:00  0.018888
2026-02-19 00:00:00+00:00  0.007094
2026-02-20 00:00:00+00:00 -0.005088
2026-02-23 00:00:00+00:00  0.004544
2026-02-24 00:00:00+00:00 -0.000544

[1257 rows x 1 columns], features=                               rvol  OVXCLS  DCOILWTICO  DHHNGSP  GASREGCOVW
session_date                                                                
2021-02-22 00:00:00+00:00  0.016316   38.57       61.67     3.16       2.549
2021-02-23 00:00:00+00:00  0.028929   39.75       61.66     2.94       2.549
2021-02-24 00:00:00+00:00  0.019238   39.58       63.21     2.80       2.549
2021-02-25 00:00:00+00:00  0.019696   40.58       63.43     2.72       2.549
2021-02-26 00:00:00

In [ ]:
model.train()